# RelayVSR qua **LightX2V** — dùng CẢ 2 GPU cho MỘT clip (Ulysses sequence parallel)

Notebook này là đường thứ hai bên cạnh `relayvsr_kaggle.ipynb` (đường ComfyUI):

| | ComfyUI (`relayvsr_kaggle.ipynb`) | **LightX2V (notebook này)** |
|---|---|---|
| Một clip dùng 2 GPU | ✖ (1 instance = 1 GPU) | ✔ **Ulysses SP**: `seq_p_size=2`, chia chuỗi token cho 2 GPU |
| Nhiều clip | ✔ 2 instance → ~2× thông lượng | ✔ tăng `--nproc_per_node`, hoặc chạy song song nhiều job |
| Cài đặt | nhẹ (pip ComfyUI + 2 file int8 5,3 GB) | **nặng**: `pip install -e .` + `triton`; `requirements.txt` đầy đủ kéo theo `sgl-kernel`, `qtorch`, `torchao`, `redis`, `pymongo`… |
| Weight | int8 của Comfy-Org (5,16 GB/cặp) | fp16 chính chủ (`wan_x2` 9,66 GB + `flash_x2` 122 MB) |
| Audio | graph giữ nguyên audio | runner SR của LightX2V **không** giữ audio → notebook tự ghép lại từ video gốc |
| Đã kiểm chứng tới đâu | tới node/API ComfyUI chạy thật | lệnh chạy/config/weight/hậu kiểm đã kiểm; **suy luận cần GPU** |

⚠️ LightX2V là framework production, tài liệu của họ khuyến nghị Docker `lightx2v/lightx2v:26062001-cu130`;
Kaggle không chạy Docker nên phải cài từ source — hãy chạy ô 2 và đọc kỹ cảnh báo ở ô 8 nếu lỗi.
Chuẩn bị: **Accelerator = GPU T4 x2**, **Internet = On**, trống ≥ 12 GB đĩa (weight 9,78 GB + output).

In [ ]:
#@title 1) Kiểm tra môi trường
import os, shutil, subprocess, sys
from pathlib import Path

!nvidia-smi | head -12
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| số GPU:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  GPU", i, torch.cuda.get_device_name(i))
print("python", sys.version.split()[0])
WORK = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".")
print("working:", WORK, "| đĩa trống:", round(shutil.disk_usage(WORK).free / 1073741824, 1), "GB")

In [ ]:
#@title 2) Repo code + clone & cài LightX2V
REPO_URL = "https://github.com/hoannguyen171/UPSCALEVIDEO.git"  #@param {type:"string"}
FULL_REQUIREMENTS = False  #@param {type:"boolean"}

CUI = WORK / "valo-upscale-gpu"
LX = WORK / "LightX2V"

if not (CUI / ".git").exists():
    !git clone --depth 1 -q "{REPO_URL}" "{CUI}"
else:
    !git -C "{CUI}" pull -q --ff-only
sys.path.insert(0, str(CUI))
import importlib, valo_upscale.lightx2v_upscale as lx
importlib.reload(lx)

lx.clone_repo(LX, log=print)
print("LightX2V:", subprocess.run(["git", "-C", str(LX), "log", "-1", "--date=short", "--format=%h %cd"],
                                  capture_output=True, text=True).stdout.strip())

for cmd in lx.install_commands(LX, full_requirements=bool(FULL_REQUIREMENTS)):
    print("$", " ".join(cmd[:6]), "...")
    subprocess.run(cmd, check=False)
print("đã cài xong — nếu bước sau báo thiếu module, bật FULL_REQUIREMENTS = True rồi chạy lại ô này")

In [ ]:
#@title 3) Tải weight fp16 (wan + flash) — có thể dùng HF_TOKEN
SCALE = 2  #@param [2, 4] {type:"raw"}
HF_TOKEN = ""  #@param {type:"string"}
WEIGHTS_SOURCE = "/kaggle/input"  #@param {type:"string"}

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("đã lấy HF_TOKEN từ Kaggle Secrets")
    except Exception:
        HF_TOKEN = ""

MODEL_DIR = LX / "models" / f"RelayVSR_X{SCALE}"
for path in lx.download_weights(int(SCALE), MODEL_DIR, token=HF_TOKEN or None,
                               source_dir=WEIGHTS_SOURCE or None, log=lambda t: print(t, flush=True)):
    print(f"  ✔ {Path(path).name} — {Path(path).stat().st_size / 1073741824:.2f} GB")
print("đủ weight:", lx.weights_ready(int(SCALE), MODEL_DIR))

In [ ]:
#@title 4) Kiểm tra môi trường chạy (thiếu gì sẽ hiện ở đây)
import json
report = lx.check_environment(LX, MODEL_DIR, int(SCALE))
print(json.dumps(report, ensure_ascii=False, indent=2))
if not report.get("cuda_available"):
    print("⚠️ Không thấy GPU — bật Accelerator = GPU T4 x2")
for name, ok in report["configs"].items():
    if not ok:
        print("⚠️ thiếu config:", name)
for name, ok in report["weights"].items():
    if not ok:
        print("⚠️ thiếu weight:", name)

## 5) Chạy upscale

Ba chế độ (đều là config chính thức của LightX2V trong `configs/relayvsr/`):

| `MODE` | Config | Khi nào dùng |
|---|---|---|
| `sp` | `relayvsr_x2_sp.json` (`parallel.seq_p_size=2`, `ulysses`) | **2 GPU cho 1 clip** — nhanh hơn |
| `single` | `relayvsr_x2.json` | 1 GPU, nhanh nhất về thủ tục |
| `block_offload` | `relayvsr_x2_block_offload.json` (`cpu_offload`) | 1 GPU ít VRAM |

`NPROC` = `seq_p_size × cfg_p_size` (sp=2, cfg=1 → 2). Ô dưới **in lệnh trước** (`DRY_RUN`) để bạn xem đúng
lệnh sẽ chạy, sau đó chạy thật và hậu kiểm kích thước + audio.

In [ ]:
#@title 5) Chạy (x2 2 GPU / x4 1 GPU / block offload ...)
INPUT = ""  #@param {type:"string"}
MODE = "sp"  #@param ["sp", "single", "block_offload"]
NPROC = 2  #@param {type:"integer"}
DTYPE = "FP16"  #@param ["FP16", "BF16"]
SEED = 42  #@param {type:"integer"}
TIMEOUT = 7200  #@param {type:"integer"}
COMPILE = True  #@param {type:"boolean"}
VIDEO_CODEC = "libx264"  #@param ["libx264", "libx265"]
DRY_RUN = True  #@param {type:"boolean"}
OUT_DIR = str(WORK / "lightx2v_out")  #@param {type:"string"}

import glob
from pathlib import Path

video = INPUT.strip()
if not video:
    found = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))
    video = found[0] if found else ""
print("video:", video or "CHƯA CÓ — điền INPUT hoặc add dataset vào /kaggle/input")

if video:
    out = Path(OUT_DIR) / f"{Path(video).stem}_relayvsr_x{SCALE}_{MODE}_{DTYPE.lower()}.mp4"
    common = dict(repo_dir=LX, model_dir=MODEL_DIR, scale=int(SCALE), mode=MODE, nproc=int(NPROC),
                  seed=int(SEED), dtype=DTYPE, timeout=int(TIMEOUT),
                  overrides={"use_compile": bool(COMPILE), "video_codec": VIDEO_CODEC})
    dry = lx.run_upscale(video, out, dry_run=True, **common)
    print("LỆNH SẼ CHẠY:\n ", " ".join(str(p) for p in dry["command"]))
    print("config:", dry["config"], "| env:", dry["env_subset"])
    if not DRY_RUN:
        result = lx.run_upscale(video, out, progress=lambda t: print(t, flush=True), **common)
        print({k: result[k] for k in ("output_path", "size_ok", "audio_ok", "frames_ok", "elapsed")})
    else:
        print("DRY_RUN = True → chưa chạy thật. Xem lệnh ổn rồi đặt DRY_RUN = False.")

In [ ]:
#@title 6) Xem kết quả
import glob
from pathlib import Path
from IPython.display import Video, display
from valo_upscale import probe

files = sorted(Path(OUT_DIR).glob("*.mp4")) if Path(OUT_DIR).exists() else []
print(f"{OUT_DIR}: {len(files)} file")
for path in files:
    info = probe.get_media_info(path)
    print(f"  • {path.name} — {path.stat().st_size / 1048576:.1f} MB | {info['width']}x{info['height']} | "
          f"{info['nb_frames']} frame | audio {info['has_audio']}")
    display(Video(str(path), embed=True, width=560))

## 7) Xử lý sự cố (đọc trước khi bỏ cuộc)

| Hiện tượng | Cách xử lý |
|---|---|
| `ModuleNotFoundError: sgl_kernel` / `qtorch` / … | Bật `FULL_REQUIREMENTS = True` ở ô 2 rồi chạy lại (nặng, có thể vẫn lệch phiên bản CUDA với Kaggle) |
| `triton` lỗi khi biên dịch kernel | Đặt `COMPILE = False` (bỏ `use_compile`) — chậm hơn nhưng dễ chạy hơn |
| Hết VRAM (T4 16 GB) | Dùng `MODE = "block_offload"`; hoặc giảm độ phân giải input; hoặc dùng đường ComfyUI int8 (`relayvsr_kaggle.ipynb`) |
| Output bị nhấp nháy/màu lạ | Kiểm `DTYPE`: T4 (Turing) **không có BF16 native** → để `FP16` |
| Video kết quả không có tiếng | Đã tự ghép từ video gốc (hàm `ensure_audio`); nếu video gốc không có tiếng thì bỏ qua |
| `size_ok: False` | LightX2V làm tròn xuống bội số 2 (YUV420) — kiểm tra theo công thức `expected_size`; hoặc `SR_RATIO` bị đặt khác |
| Muốn nhanh hơn nữa | Tăng `NPROC` nếu có nhiều GPU (Kaggle T4 x2 → tối đa 2); hoặc chạy nhiều clip song song |

**Chưa kiểm chứng được ở máy dev:** phần suy luận RelayVSR (cần GPU NVIDIA). Đã kiểm: config chính thức có
thật và đúng khối `parallel`; lệnh chạy dựng đúng (đối chiếu `scripts/dist_infer/*.sh`); tải weight đúng
repo + đúng tên file + có token + dùng lại bản sẵn có; `expected_size`; ghép audio bằng ffmpeg thật;
và **chạy trọn đường ống bằng repo LightX2V giả** (subprocess thật → ra mp4 đúng 2×, ghép tiếng, dọn file tạm,
bắt log). Lần chạy đầu trên Kaggle hãy thử **1 clip ngắn** và để `DRY_RUN = True` xem lệnh trước.

In [ ]:
#@title 8) Liệt kê / nén kết quả (quét cả thư mục con)
import shutil
from pathlib import Path
from valo_upscale import probe

out_dir = Path(OUT_DIR)
# rglob vì ô 6b (chạy song song) ghi vào thư mục con: <OUT_DIR>/song_song/...
files = sorted(out_dir.rglob("*.mp4")) if out_dir.exists() else []

per_folder = {}
for f in files:
    key = str(f.parent.relative_to(out_dir)) or "."
    per_folder.setdefault(key, []).append(f)

print(f"{out_dir}: {len(files)} file trong {len(per_folder)} thư mục")
for folder, items in sorted(per_folder.items()):
    print(f"  [thư mục] {folder}: {len(items)} file")
for f in files:
    info = probe.get_media_info(f)
    print(f"    - {f.relative_to(out_dir)} — {f.stat().st_size / 1048576:.1f} MB | "
          f"{info['width']}x{info['height']} | {info['nb_frames']} frame | audio {info['has_audio']}")

if files:
    zip_path = shutil.make_archive(str(out_dir.parent / "relayvsr_ket_qua"), "zip", root_dir=str(out_dir))
    print("ZIP:", zip_path, f"({Path(zip_path).stat().st_size / 1048576:.1f} MB)")
    if not Path("/kaggle/working").exists():
        try:
            from google.colab import files as colab_files
            colab_files.download(zip_path)
        except Exception as exc:
            print("(không phải Colab nên không tự tải về:", exc, ")")
else:
    print("Chưa có video nào — chạy ô 6 (một clip) hoặc ô 6b (nhiều clip) trước.")